# Claim Rx Monthly Features — Cohort-Based

Refactored from `01_claim_rx_monthly` (NBA 3.0) to support **configurable cohort windows with offset**.

* **Offset = 0** reproduces the original calendar-month behavior
* **Offset = 7 / 14 / 21** slides all windows forward by that many days
* **Lookback windows** (1, 3, 6, 9, 12 months) are date-range-based, not row-based

Source notebook: `/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/03_Feature_Engineering/03_HCP_Level_Features/01_claim_rx_monthly`

## Part 1 — Reverse Engineering of `01_claim_rx_monthly`

### Pipeline Overview

```
df_rx (daily claims)    pat_new_and_total_treatments (monthly)
       │                              │
       ├─ date_trunc(‘MM’, SVC_DT) ─────────┤
       │   → SVC_DT_M (month grain)    │
       │                              │
       ├─ 8 Feature Groups (pivot+count per NPI per month)
       │   2.1  Treatment Category    ─┐
       │   2.2  Brand Level           │
       │   2.3  Rejection             │
       │   2.4  Duration (Tx×DaysSupply)│
       │   2.5  Refill (Tx×RefillFlag)│
       │   2.6  Payer Plan (Tx×Payment)│
       │   2.7  Patient Attributes    │
       │   2.8  Patient Treatment     │
       │                              │
       ├─ Combine (left-join on NPI,month) + SGLT2_WO_INVOKANA derived cols
       │
       ├─ step2_fill_missing → cross-join NPI×months, left-join features
       │
       ├─ step3_lag → Window(rowsBetween) for SUM in last [1,3,6,12] months
       │               + RUNNING totals
       │
       ├─ PCT features (feature / total_claims, per window)
       │
       ├─ AVG features (avg_age, avg_treatment_per_patient) with step3_lag_avg
       │
       ├─ Difference features (_IN_LAST_3_6_MONTH = last_6 − last_3)
       │   Ratio features (prior_3 / last_3)
       │
       ├─ Filter SVC_DT_M >= 2022-01-01
       │
       └─ Write → claim_rx_monthly (Delta)
              → Join NPI→BH_ID → claim_rx_monthly_provider (Delta)
```

**Output grain**: One row per (NPI_NUMBER, SVC_DT_M) — ~3.3M rows  
**Output location**: `/Volumes/.../notebook_output/claim_rx_monthly`

### Data Sources

| Source | Path | Grain | Key Columns Used |
|--------|------|-------|------------------|
| Rx claims | `.../query_output/Rx_01_new` | 1 row per claim | NPI_NUMBER, SVC_DT, TREATMENT_CATEGORY, Category, REJECTION_FLAG, DAYS_SUPPLY, REFILL_FLAG, METHOD_OF_PAYMENT, PATIENT_ID, PATIENT_BIRTH_YEAR, PATIENT_GENDER |
| Patient treatments | `.../query_output/pat_new_and_total_treatments` | 1 row per (PATIENT_ID, month) | PATIENT_ID, SVC_DT_M, TOTAL_TREATMENTS, TOTAL_NEW_TREATMENTS |
| NPI mapping | `.../kerendia_nba_3_volume/bh_id_npi_mapping` | 1 row per NPI | NPI_NUMBER, BH_ID |

### Feature Groups (Step 2 in original)

All feature groups follow the same pattern: **groupBy(NPI, month) → pivot(dimension) → count claims → fill zeros**.

Column naming convention: `TOTAL_RX_CLAIMS_{PIVOT_VALUE}` (or `TOTAL_REJECTED_RX_CLAIMS_` for rejections).

| # | Group | Pivot Column(s) | Filter | Prefix | Cell |
|---|-------|----------------|--------|--------|------|
| 2.1 | Treatment Category | TREATMENT_CATEGORY | none | `TOTAL_RX_CLAIMS_` | 12 |
| 2.2 | Brand Level | Category | Category IN (‘INVOKANA 100MG’, ‘INVOKANA 300MG’, ‘FARXIGA’, ‘JARDIANCE’, ‘SPIRONOLACTONE-MRA’, ‘Diuretic + MRA’, ‘DPP-IV’, ‘GLP-1’) | `TOTAL_RX_CLAIMS_` | 15 |
| 2.3 | Rejection | TREATMENT_CATEGORY + Category (same filters as 2.1+2.2) | REJECTION_FLAG = 1 | `TOTAL_REJECTED_RX_CLAIMS_` | 18 |
| 2.4 | Duration | TREATMENT_CATEGORY×DAYS_SUPPLY + Category×DAYS_SUPPLY | none (DAYS_SUPPLY bucketed to 30/60/90/Not/Unknown) | `TOTAL_RX_CLAIMS_` | 20–28 |
| 2.5 | Refill | TREATMENT_CATEGORY×REFILL_FLAG + Category×REFILL_FLAG + REFILL_FLAG alone | none | `TOTAL_RX_CLAIMS_` | 30 |
| 2.6 | Payer Plan | TREATMENT_CATEGORY×METHOD_OF_PAYMENT + Category×METHOD_OF_PAYMENT + METHOD_OF_PAYMENT alone | none (null → Unknown) | `TOTAL_RX_CLAIMS_` | 32 |
| 2.7 | Patient Attributes | (Tx×AGE_BIN, Cat×AGE_BIN, AGE_BIN, Tx×GENDER, Cat×GENDER, GENDER) | none (AGE bucketed to 10-yr bins, Gender null → Unknown) | `TOTAL_RX_CLAIMS_` | 35 |
| 2.8 | Patient Treatment | TOTAL_ACTIVE_TREATMENTS bucket, TOTAL_NEW_TREATMENTS bucket | none (joined with pat_new_and_total_treatments) | `TOTAL_RX_CLAIMS_PATIENT_WITH_` | 37–38 |

### Derived Features (after combine)

| Feature Pattern | Logic | Cell |
|----------------|-------|------|
| `SGLT2_WO_INVOKANA_300MG` variants | SGLT2 − INVOKANA_300MG (for counts, duration, rejection, age, gender) | 40 |
| `_IN_LAST_{1,3,6,12}_MONTH` | Rolling SUM via `Window.rowsBetween(-i+1, 0)` on monthly rows | 44–45 |
| `RUNNING_` prefix | Unbounded preceding SUM | 45 |
| `PCT_` prefix | feature / TOTAL_RX_CLAIMS (per window) | 48–52 |
| `AVG_PATIENT_AGE_IN_LAST_{i}_MONTH` | Rolling AVG via `Window.rowsBetween(-i, 0)` | 55–56 |
| `AVG_TREATMENT_PER_PATIENT_IN_LAST_{i}_MONTH` | Rolling AVG | 55–56 |
| `TOTAL_PATIENT_COUNT_IN_LAST_{i}_MONTH` | Rolling AVG (oddly — averages monthly counts) | 55–56 |
| `_IN_LAST_3_6_MONTH` | last_6 − last_3 ("prior 3 months") | 60 |
| `RATIO_..._IN_PRIOR_3_LAST_3_MONTH` | prior_3 / last_3 (safe div) | 61 |

### Where Monthly Logic Is Hardcoded (⭐ Refactor Targets)

| Location | Code | What It Does | Impact |
|----------|------|-------------|--------|
| **Cell 7** | `date_trunc('MM', SVC_DT)` | Truncates daily claims to first-of-month | ⭐ Must keep daily `SVC_DT` for offset > 0; month truncation only valid for offset = 0 |
| **step2_fill_missing** (helper) | `sequence(min_date, max_date, interval 1 month)` | Creates monthly grid for cross-join | ⭐ Replaced entirely — no grid needed when computing per date-range window |
| **step3_lag** (helper) | `Window.rowsBetween(-i+1, 0)` | Rolling SUM over last i months (row = month) | ⭐ Replaced by date-range filter + aggregate per window |
| **step3_lag_avg** (helper) | `Window.rowsBetween(-i, 0)` | Rolling AVG over last i+1 months | ⭐ Same replacement |
| **Cell 62** | `filter(to_date('SVC_DT_M') >= '2022-01-01')` | Hardcoded start date filter | ⭐ Replaced by cohort config |
| **Cell 37** | `join(pat_new_and_total_treatments, on=['PATIENT_ID', 'SVC_DT_M'])` | Joins monthly patient treatments | ✅ Resolved — `assign_cohort_month()` aligns the join key with the patient journey's offset-based SVC_DT_M |
| **Cell 35** | `F.year(col('SVC_DT_M')) - PATIENT_BIRTH_YEAR` | Age from month year | Minor — can use `SVC_DT` year instead |

### ⚠️ Flags for Wenjie

1. ~~**`pat_new_and_total_treatments`** month alignment~~ **RESOLVED**: `assign_cohort_month()` (added to helper functions) maps claim dates to offset-aligned cohort months, matching the patient journey pipeline's `SVC_DT_M`. Applied in cells 18 (windowed patient treatment), 19 (RUNNING patient treatment), and 20 (AVG monthly aggregation). No daily-grain version needed.
2. **`00_write_input_files`**: The `Rx_01_new` source table already has daily `SVC_DT` — no change needed. But if the write notebook creates any monthly aggregates, those may need updating.
3. **`RUNNING_` features**: In the cohort model, unbounded-preceding running totals don’t have a clear interpretation (running from when?). Consider dropping or redefining as running total within a fixed horizon (e.g., 24 months).
4. **`step2_fill_missing` and `step3_lag/step3_lag_avg`** in `00_step23_function` are no longer needed for this notebook. Other notebooks (02–06) still use them until they are also refactored.

## Part 2 — Cohort-Based Design

### How the Offset Works

```
Calendar:     Jan     Feb     Mar     Apr     May
              |-------|-------|-------|-------|

Cohort = Mar’26, Offset = 0:
  Cutoff = 31 Mar
  Last 1m:  1 Mar ─────── 31 Mar
  Last 3m:  1 Jan ─────────────── 31 Mar

Cohort = Mar’26, Offset = 7:
  Cutoff = 7 Apr
  Last 1m:  7 Mar ─────── 7 Apr
  Last 3m:  7 Jan ─────────────── 7 Apr
```

The entire lookback slides forward by the offset. At offset = 0, windows align with calendar months.

### Output Grain

**Old**: One row per (NPI_NUMBER, SVC_DT_M) — one per calendar month  
**New**: One row per (NPI_NUMBER, COHORT_ID) — one per cohort run

For training, generate multiple cohorts (C1 = Mar’26, C2 = Feb’26, …). For scoring, generate the current cohort only.

### Feature Naming Convention (preserved)

Base window features keep the same `_IN_LAST_{N}_MONTH` suffix to maintain downstream compatibility. The cohort config records which dates each window actually covers.

### What Changed vs. Original

| Aspect | Original (3.0) | Cohort-Based (3.1) |
|--------|---------------|--------------------|
| Time grain | Calendar month | Date range from cutoff |
| Aggregation | `date_trunc` → monthly pivot → row-window | Date-range filter → direct pivot per window |
| Window definition | `rowsBetween(-i+1, 0)` on monthly rows | `SVC_DT BETWEEN cutoff-Nm AND cutoff` |
| Fill missing | Cross-join NPI×months | NPI universe from max window; outer-join per window |
| RUNNING features | Unbounded preceding sum | ⚠️ Dropped — redefine if needed |
| Parameters | None (hardcoded) | COHORT_MONTH, OFFSET_DAYS, LOOKBACK_WINDOWS |

## Part 3 — Implementation

### 0. Configuration

**Cell order note**: The `%run` cells MUST execute before the cohort parameters cell.
`00_connection` may call `restartPython()`, which clears all prior variables.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../00_cohort_config"

In [0]:
# All config (COHORT_MONTH, OFFSET_DAYS, CUTOFF_DATE, WINDOW_BOUNDS,
# LOOKBACK_WINDOWS, NPI_COL, SVC_COL, VALIDATION_MODE, apply_hcp_filter,
# apply_npi_filter, assign_cohort_month, etc.) comes from 00_cohort_config.

print(f"Cohort: {COHORT_MONTH} | Offset: {OFFSET_DAYS} | "
      f"Cutoff: {CUTOFF_DATE.strftime('%Y-%m-%d')} | "
      f"Validation: {VALIDATION_MODE}")

In [0]:
# ---------- Load raw Rx claims (daily level) ----------
df_rx = spark.read.format("delta").load(
    '/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/'
    'kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/Rx_01_new'
)

# Same cleanup as original
df_rx = df_rx.withColumn(
    'TREATMENT_CATEGORY', F.regexp_replace('TREATMENT_CATEGORY', r'[\s]*$', '')
)

# pat_new_and_total_treatments is monthly (keyed on offset-aligned SVC_DT_M from NB06).
# We use assign_cohort_month() for the join so months align at any offset.
# See cells 18 (windowed) and 19 (RUNNING) for the join logic.
pat_new_and_total_treatments = spark.read.format("delta").load(
    '/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/'
    'kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/pat_new_and_total_treatments'
)

# ---------- Pre-filter to max lookback window ----------
# Use AVG_WINDOW_BOUNDS (1 month wider when AVG_EXTRA_MONTH=True)
# so that the AVG features cell has enough data.
_max_window = builtins.max(LOOKBACK_WINDOWS)
_max_start  = AVG_WINDOW_BOUNDS[_max_window]["start"]
_end        = AVG_WINDOW_BOUNDS[_max_window]["end"]

df_rx_filtered = df_rx.filter(
    (F.col("SVC_DT") > F.lit(_max_start)) &
    (F.col("SVC_DT") <= F.lit(_end))
)
# NOTE: .cache() removed — not supported on serverless compute

print(f"Filtered claims to ({_max_start}, {_end}]")
print(f"Row count: {df_rx_filtered.count():,}")

### 1. Build Feature Groups

Each group uses `build_feature_group` with date-range filters instead of monthly pivot + row-window.

In [0]:
BRAND_FILTER = "Category IN ('INVOKANA 100MG', 'INVOKANA 300MG', 'FARXIGA', 'JARDIANCE', 'SPIRONOLACTONE-MRA', 'Diuretic + MRA', 'DPP-IV', 'GLP-1')"

# ---- NPI universe (windowed claims only — widened with df_running NPIs in cell 20) ----
# Filter null NPIs — source data has ~16.7M null-NPI claims that are useless
npi_universe = df_rx_filtered.filter(F.col(NPI_COL).isNotNull()).select(NPI_COL).distinct()
print(f"NPI universe: {npi_universe.count():,} providers")

# ---- Total Rx Claims per window (denominator for PCT) ----
df_total = build_total_col(df_rx_filtered, NPI_COL, SVC_COL, WINDOW_BOUNDS)

# ---- 2.1 Treatment Category ----
df_category = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col="TREATMENT_CATEGORY",
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS
)

# ---- 2.2 Brand Level ----
df_brand = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col="Category",
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    extra_filter=BRAND_FILTER
)

print(f"Treatment Category cols: {len(df_category.columns) - 1}")
print(f"Brand cols: {len(df_brand.columns) - 1}")

In [0]:
# ---- 2.3 Rejection (Treatment Category) ----
df_rej_category = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col="TREATMENT_CATEGORY",
    prefix="TOTAL_REJECTED_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    extra_filter="REJECTION_FLAG = 1"
)

# ---- 2.3 Rejection (Brand) ----
df_rej_brand = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col="Category",
    prefix="TOTAL_REJECTED_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    extra_filter=f"REJECTION_FLAG = 1 AND {BRAND_FILTER}"
)

df_rejection = df_rej_category.join(df_rej_brand, on=NPI_COL, how="outer").na.fill(0)

# ---- 2.4 Duration — bucket DAYS_SUPPLY, then pivot on Tx×Supply and Cat×Supply ----
df_rx_dur = df_rx_filtered.withColumn(
    "DAYS_SUPPLY_BIN",
    F.when(F.col("DAYS_SUPPLY").rlike(r"^\s*30(\.0+)?\s*$"), "30_DAYS_SUPPLY")
    .when(F.col("DAYS_SUPPLY").rlike(r"^\s*60(\.0+)?\s*$"), "60_DAYS_SUPPLY")
    .when(F.col("DAYS_SUPPLY").rlike(r"^\s*90(\.0+)?\s*$"), "90_DAYS_SUPPLY")
    .when(F.col("DAYS_SUPPLY").isNull(), "DAYS_SUPPLY_Unknown")
    .otherwise("Not_30_60_90_Days_Supply")
)

df_dur_category = build_feature_group(
    df_rx_dur, NPI_COL, SVC_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    combine_cols=["TREATMENT_CATEGORY", "DAYS_SUPPLY_BIN"]
)

df_dur_brand = build_feature_group(
    df_rx_dur, NPI_COL, SVC_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "DAYS_SUPPLY_BIN"]
)

df_duration = df_dur_category.join(df_dur_brand, on=NPI_COL, how="outer").na.fill(0)

print(f"Rejection cols: {len(df_rejection.columns) - 1}")
print(f"Duration cols: {len(df_duration.columns) - 1}")

In [0]:
# ---- 2.5 Refill ----
# Old notebook had brand × refill + refill alone only (no treatment × refill).

# Brand × Refill
df_refill_brand = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "REFILL_FLAG"]
)

# Refill alone
df_refill_alone = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col="REFILL_FLAG",
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS
)

df_refill = (
    df_refill_alone
    .join(df_refill_brand, on=NPI_COL, how="outer")
    .na.fill(0)
)

# ---- 2.6 Payer Plan ----
# Prep: null payment → Unknown
df_rx_pay = df_rx_filtered.withColumn(
    "METHOD_OF_PAYMENT",
    F.when(F.col("METHOD_OF_PAYMENT").isNull(), "METHOD_OF_PAYMENT_Unknown")
    .otherwise(F.col("METHOD_OF_PAYMENT"))
)

df_payer_cat = build_feature_group(
    df_rx_pay, NPI_COL, SVC_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    combine_cols=["TREATMENT_CATEGORY", "METHOD_OF_PAYMENT"]
)

df_payer_brand = build_feature_group(
    df_rx_pay, NPI_COL, SVC_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "METHOD_OF_PAYMENT"]
)

df_payer_alone = build_feature_group(
    df_rx_pay, NPI_COL, SVC_COL,
    pivot_col="METHOD_OF_PAYMENT",
    prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS
)

df_payer = (
    df_payer_alone
    .join(df_payer_cat, on=NPI_COL, how="outer")
    .join(df_payer_brand, on=NPI_COL, how="outer")
    .na.fill(0)
)

print(f"Refill cols: {len(df_refill.columns) - 1}")
print(f"Payer cols: {len(df_payer.columns) - 1}")

In [0]:
# ---- 2.7 Patient Attributes ----
# Prep: compute AGE_BIN and clean PATIENT_GENDER (same logic as original)
df_rx_pat = (
    df_rx_filtered
    .withColumn("AGE",
        F.when((F.col("PATIENT_BIRTH_YEAR") == 0) | (F.col("PATIENT_BIRTH_YEAR").isNull()), None)
        .otherwise(F.year(F.col("SVC_DT")) - F.col("PATIENT_BIRTH_YEAR"))
    )
    .withColumn("AGE_ROUNDED", (F.floor(F.col("AGE") / 10) * 10))
    .withColumn("AGE_BIN",
        F.when((F.col("PATIENT_BIRTH_YEAR") == 0) | (F.col("PATIENT_BIRTH_YEAR").isNull()), "AGE_GROUP_IGNORE")
        .otherwise(
            F.when(F.col("AGE_ROUNDED") >= 80, "AGE_GROUP_ABOVE_80")
            .otherwise(
                F.concat(F.lit("AGE_GROUP_"),
                         F.col("AGE_ROUNDED").cast("int").cast("string"),
                         F.lit("_"),
                         (F.col("AGE_ROUNDED") + 10).cast("int").cast("string"))
            )
        )
    )
    .withColumn("PATIENT_GENDER",
        F.when((F.col("PATIENT_GENDER") == "null") | (F.col("PATIENT_GENDER").isNull()), "PATIENT_GENDER_Unknown")
        .otherwise(F.col("PATIENT_GENDER"))
    )
)

# Age pivots: Tx×Age, Cat×Age, Age alone
df_pat_age_cat = build_feature_group(
    df_rx_pat, NPI_COL, SVC_COL, pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_", window_bounds=WINDOW_BOUNDS,
    combine_cols=["TREATMENT_CATEGORY", "AGE_BIN"]
)
df_pat_age_brand = build_feature_group(
    df_rx_pat, NPI_COL, SVC_COL, pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_", window_bounds=WINDOW_BOUNDS,
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "AGE_BIN"]
)
df_pat_age_alone = build_feature_group(
    df_rx_pat, NPI_COL, SVC_COL, pivot_col="AGE_BIN",
    prefix="TOTAL_RX_CLAIMS_", window_bounds=WINDOW_BOUNDS
)

# Gender pivots: Tx×Gender, Cat×Gender, Gender alone
df_pat_gen_cat = build_feature_group(
    df_rx_pat, NPI_COL, SVC_COL, pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_", window_bounds=WINDOW_BOUNDS,
    combine_cols=["TREATMENT_CATEGORY", "PATIENT_GENDER"]
)
df_pat_gen_brand = build_feature_group(
    df_rx_pat, NPI_COL, SVC_COL, pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_", window_bounds=WINDOW_BOUNDS,
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "PATIENT_GENDER"]
)
df_pat_gen_alone = build_feature_group(
    df_rx_pat, NPI_COL, SVC_COL, pivot_col="PATIENT_GENDER",
    prefix="TOTAL_RX_CLAIMS_", window_bounds=WINDOW_BOUNDS
)

df_patient = (
    df_pat_age_alone
    .join(df_pat_age_cat, on=NPI_COL, how="outer")
    .join(df_pat_age_brand, on=NPI_COL, how="outer")
    .join(df_pat_gen_alone, on=NPI_COL, how="outer")
    .join(df_pat_gen_cat, on=NPI_COL, how="outer")
    .join(df_pat_gen_brand, on=NPI_COL, how="outer")
    .na.fill(0)
)

print(f"Patient attribute cols: {len(df_patient.columns) - 1}")

In [0]:
# ---- 2.8 Patient Treatment (active/new treatment buckets) ----
# pat_new_and_total_treatments is monthly, keyed on offset-aligned SVC_DT_M from NB06.
# We use assign_cohort_month() to match the patient journey's month assignment,
# so claims near boundaries (e.g. Jan 5 with offset=7) map to the correct cohort month.

def build_patient_treatment_features(df_rx, pat_treatments, npi_col, svc_dt_col, window_bounds, brand_filter):
    """Build patient treatment bucket features across all windows."""
    window_dfs = []
    
    for w in sorted(window_bounds.keys()):
        bounds = window_bounds[w]
        df_w = filter_to_window(df_rx, svc_dt_col, bounds["start"], bounds["end"])
        
        # Join with patient treatments on offset-aligned cohort month
        df_w = df_w.withColumn("_SVC_DT_M", assign_cohort_month(F.col(svc_dt_col)))
        df_w = df_w.join(
            pat_treatments,
            (df_w["PATIENT_ID"] == pat_treatments["PATIENT_ID"]) &
            (df_w["_SVC_DT_M"] == pat_treatments["SVC_DT_M"]),
            how="left"
        ).drop(pat_treatments["PATIENT_ID"]).drop(pat_treatments["SVC_DT_M"])
        
        # Bucket treatment counts (same as original)
        df_w = df_w.withColumn("TOTAL_ACTIVE_TREATMENTS",
            F.when((F.col("TOTAL_TREATMENTS") < 1) | (F.col("TOTAL_TREATMENTS").isNull()), "NO_ACTIVE_TREATMENT")
            .otherwise(F.when(F.col("TOTAL_TREATMENTS") == 1, "1_ACTIVE_TREATMENT")
            .otherwise(F.when(F.col("TOTAL_TREATMENTS") < 3, "2_OR_3_ACTIVE_TREATMENTS")
            .otherwise("3_MORE_ACTIVE_TREATMENTS")))
        ).withColumn("TOTAL_NEW_TREATMENTS",
            F.when((F.col("TOTAL_NEW_TREATMENTS") < 1) | (F.col("TOTAL_NEW_TREATMENTS").isNull()), "NO_NEW_TREATMENT")
            .otherwise(F.when(F.col("TOTAL_NEW_TREATMENTS") == 1, "1_NEW_TREATMENT")
            .otherwise(F.when(F.col("TOTAL_NEW_TREATMENTS") < 3, "2_OR_3_NEW_TREATMENTS")
            .otherwise("3_MORE_NEW_TREATMENTS")))
        )
        
        suffix = f"_IN_LAST_{w}_MONTH"
        
        # Active treatments pivot
        df_active = pivoted_counts(df_w, npi_col, "TOTAL_ACTIVE_TREATMENTS",
                                    "TOTAL_RX_CLAIMS_PATIENT_WITH_", suffix)
        # New treatments pivot
        df_new = pivoted_counts(df_w, npi_col, "TOTAL_NEW_TREATMENTS",
                                 "TOTAL_RX_CLAIMS_PATIENT_WITH_", suffix)
        
        df_combined = df_active.join(df_new, on=npi_col, how="outer").na.fill(0)
        window_dfs.append(df_combined)
    
    # Join all windows
    result = window_dfs[0]
    for df_next in window_dfs[1:]:
        result = result.join(df_next, on=npi_col, how="outer")
    return result.na.fill(0)

df_patient_treatment = build_patient_treatment_features(
    df_rx_filtered, pat_new_and_total_treatments,
    NPI_COL, SVC_COL, WINDOW_BOUNDS, BRAND_FILTER
)

print(f"Patient treatment cols: {len(df_patient_treatment.columns) - 1}")

In [0]:
# ---- RUNNING_ Features from full Rx history through the cohort cutoff ----
# This reproduces the old notebook's unbounded-preceding RUNNING_ columns.
# Important: this uses the full Rx_01_new history (df_rx), not df_rx_filtered.
# It also excludes null NPIs so the widened doctor universe is based only on real providers.

cutoff_lit = F.to_date(F.lit(CUTOFF_DATE.strftime("%Y-%m-%d")))

df_rx_running = (
    df_rx
    .filter(F.col(NPI_COL).isNotNull())
    .filter(F.col(SVC_COL) <= cutoff_lit)
)


def build_running_total(df_in, npi_col):
    return (
        df_in.groupBy(npi_col)
        .agg(F.count(F.lit(1)).alias("RUNNING_TOTAL_RX_CLAIMS"))
    )


def build_running_feature_group(df_in, npi_col, pivot_col, prefix, extra_filter=None, combine_cols=None):
    """Build one RUNNING_ feature group over full history through CUTOFF_DATE."""
    df_work = df_in

    if extra_filter:
        df_work = df_work.filter(extra_filter)

    actual_pivot = pivot_col
    if combine_cols:
        df_work = df_work.withColumn(
            "_pivot_combined",
            F.concat_ws("_", *[F.col(c) for c in combine_cols])
        )
        actual_pivot = "_pivot_combined"

    return pivoted_counts(df_work, npi_col, actual_pivot, f"RUNNING_{prefix}", "")


# ---- Base running groups ----
df_running_total = build_running_total(df_rx_running, NPI_COL)

df_running_category = build_running_feature_group(
    df_rx_running, NPI_COL,
    pivot_col="TREATMENT_CATEGORY",
    prefix="TOTAL_RX_CLAIMS_"
)

df_running_brand = build_running_feature_group(
    df_rx_running, NPI_COL,
    pivot_col="Category",
    prefix="TOTAL_RX_CLAIMS_",
    extra_filter=BRAND_FILTER
)

# Rejections: treatment + brand
df_running_rej_category = build_running_feature_group(
    df_rx_running, NPI_COL,
    pivot_col="TREATMENT_CATEGORY",
    prefix="TOTAL_REJECTED_RX_CLAIMS_",
    extra_filter="REJECTION_FLAG = 1"
)

df_running_rej_brand = build_running_feature_group(
    df_rx_running, NPI_COL,
    pivot_col="Category",
    prefix="TOTAL_REJECTED_RX_CLAIMS_",
    extra_filter=f"REJECTION_FLAG = 1 AND {BRAND_FILTER}"
)

df_running_rejection = (
    df_running_rej_category
    .join(df_running_rej_brand, on=NPI_COL, how="outer")
    .na.fill(0)
)

# Duration: treatment × days supply + brand × days supply
df_rx_running_dur = df_rx_running.withColumn(
    "DAYS_SUPPLY_BIN",
    F.when(F.col("DAYS_SUPPLY").rlike(r"^\s*30(\.0+)?\s*$"), "30_DAYS_SUPPLY")
    .when(F.col("DAYS_SUPPLY").rlike(r"^\s*60(\.0+)?\s*$"), "60_DAYS_SUPPLY")
    .when(F.col("DAYS_SUPPLY").rlike(r"^\s*90(\.0+)?\s*$"), "90_DAYS_SUPPLY")
    .when(F.col("DAYS_SUPPLY").isNull(), "DAYS_SUPPLY_Unknown")
    .otherwise("Not_30_60_90_Days_Supply")
)

df_running_dur_category = build_running_feature_group(
    df_rx_running_dur, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    combine_cols=["TREATMENT_CATEGORY", "DAYS_SUPPLY_BIN"]
)

df_running_dur_brand = build_running_feature_group(
    df_rx_running_dur, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "DAYS_SUPPLY_BIN"]
)

df_running_duration = (
    df_running_dur_category
    .join(df_running_dur_brand, on=NPI_COL, how="outer")
    .na.fill(0)
)

# Refill: old notebook had brand × refill + refill alone only (no treatment × refill)
df_running_refill_brand = build_running_feature_group(
    df_rx_running, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "REFILL_FLAG"]
)

df_running_refill_alone = build_running_feature_group(
    df_rx_running, NPI_COL,
    pivot_col="REFILL_FLAG",
    prefix="TOTAL_RX_CLAIMS_"
)

df_running_refill = (
    df_running_refill_alone
    .join(df_running_refill_brand, on=NPI_COL, how="outer")
    .na.fill(0)
)

# Payer: treatment × payer + brand × payer + payer alone
df_rx_running_pay = df_rx_running.withColumn(
    "METHOD_OF_PAYMENT",
    F.when(F.col("METHOD_OF_PAYMENT").isNull(), "METHOD_OF_PAYMENT_Unknown")
    .otherwise(F.col("METHOD_OF_PAYMENT"))
)

df_running_payer_category = build_running_feature_group(
    df_rx_running_pay, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    combine_cols=["TREATMENT_CATEGORY", "METHOD_OF_PAYMENT"]
)

df_running_payer_brand = build_running_feature_group(
    df_rx_running_pay, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "METHOD_OF_PAYMENT"]
)

df_running_payer_alone = build_running_feature_group(
    df_rx_running_pay, NPI_COL,
    pivot_col="METHOD_OF_PAYMENT",
    prefix="TOTAL_RX_CLAIMS_"
)

df_running_payer = (
    df_running_payer_alone
    .join(df_running_payer_category, on=NPI_COL, how="outer")
    .join(df_running_payer_brand, on=NPI_COL, how="outer")
    .na.fill(0)
)

# Patient attributes: age bins + gender, same preparation as the feature-group cell
# Age uses SVC_DT year (cohort notebook semantics).
df_rx_running_pat = (
    df_rx_running
    .withColumn(
        "AGE",
        F.when((F.col("PATIENT_BIRTH_YEAR") == 0) | (F.col("PATIENT_BIRTH_YEAR").isNull()), None)
        .otherwise(F.year(F.col("SVC_DT")) - F.col("PATIENT_BIRTH_YEAR"))
    )
    .withColumn("AGE_ROUNDED", F.floor(F.col("AGE") / 10) * 10)
    .withColumn(
        "AGE_BIN",
        F.when((F.col("PATIENT_BIRTH_YEAR") == 0) | (F.col("PATIENT_BIRTH_YEAR").isNull()), "AGE_GROUP_IGNORE")
        .otherwise(
            F.when(F.col("AGE_ROUNDED") >= 80, "AGE_GROUP_ABOVE_80")
            .otherwise(
                F.concat(
                    F.lit("AGE_GROUP_"),
                    F.col("AGE_ROUNDED").cast("int").cast("string"),
                    F.lit("_"),
                    (F.col("AGE_ROUNDED") + 10).cast("int").cast("string")
                )
            )
        )
    )
    .withColumn(
        "PATIENT_GENDER",
        F.when((F.col("PATIENT_GENDER") == "null") | (F.col("PATIENT_GENDER").isNull()), "PATIENT_GENDER_Unknown")
        .otherwise(F.col("PATIENT_GENDER"))
    )
)

df_running_pat_age_category = build_running_feature_group(
    df_rx_running_pat, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    combine_cols=["TREATMENT_CATEGORY", "AGE_BIN"]
)

df_running_pat_age_brand = build_running_feature_group(
    df_rx_running_pat, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "AGE_BIN"]
)

df_running_pat_age_alone = build_running_feature_group(
    df_rx_running_pat, NPI_COL,
    pivot_col="AGE_BIN",
    prefix="TOTAL_RX_CLAIMS_"
)

df_running_pat_gender_category = build_running_feature_group(
    df_rx_running_pat, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    combine_cols=["TREATMENT_CATEGORY", "PATIENT_GENDER"]
)

df_running_pat_gender_brand = build_running_feature_group(
    df_rx_running_pat, NPI_COL,
    pivot_col=None,
    prefix="TOTAL_RX_CLAIMS_",
    extra_filter=BRAND_FILTER,
    combine_cols=["Category", "PATIENT_GENDER"]
)

df_running_pat_gender_alone = build_running_feature_group(
    df_rx_running_pat, NPI_COL,
    pivot_col="PATIENT_GENDER",
    prefix="TOTAL_RX_CLAIMS_"
)

df_running_patient = (
    df_running_pat_age_alone
    .join(df_running_pat_age_category, on=NPI_COL, how="outer")
    .join(df_running_pat_age_brand, on=NPI_COL, how="outer")
    .join(df_running_pat_gender_alone, on=NPI_COL, how="outer")
    .join(df_running_pat_gender_category, on=NPI_COL, how="outer")
    .join(df_running_pat_gender_brand, on=NPI_COL, how="outer")
    .na.fill(0)
)

# Patient treatment buckets: join claims to the monthly patient-treatment table on claim month.
# Rename join keys on the right side to avoid AMBIGUOUS_REFERENCE on Spark Connect
# (.drop(df["col"]) is unreliable when the plan contains prior broadcast joins).
_pat_tx = (
    pat_new_and_total_treatments
    .withColumnRenamed("PATIENT_ID", "_pat_PATIENT_ID")
    .withColumnRenamed("SVC_DT_M", "_pat_SVC_DT_M")
)
df_running_tx_joined = (
    df_rx_running
    .withColumn("_SVC_DT_M", assign_cohort_month(F.col(SVC_COL)))
    .join(
        _pat_tx,
        (F.col("PATIENT_ID") == F.col("_pat_PATIENT_ID")) &
        (F.col("_SVC_DT_M") == F.col("_pat_SVC_DT_M")),
        how="left"
    )
    .drop("_pat_PATIENT_ID", "_pat_SVC_DT_M")
    .withColumn(
        "TOTAL_ACTIVE_TREATMENTS",
        F.when((F.col("TOTAL_TREATMENTS") < 1) | (F.col("TOTAL_TREATMENTS").isNull()), "NO_ACTIVE_TREATMENT")
        .otherwise(
            F.when(F.col("TOTAL_TREATMENTS") == 1, "1_ACTIVE_TREATMENT")
            .otherwise(
                F.when(F.col("TOTAL_TREATMENTS") < 3, "2_OR_3_ACTIVE_TREATMENTS")
                .otherwise("3_MORE_ACTIVE_TREATMENTS")
            )
        )
    )
    .withColumn(
        "TOTAL_NEW_TREATMENTS",
        F.when((F.col("TOTAL_NEW_TREATMENTS") < 1) | (F.col("TOTAL_NEW_TREATMENTS").isNull()), "NO_NEW_TREATMENT")
        .otherwise(
            F.when(F.col("TOTAL_NEW_TREATMENTS") == 1, "1_NEW_TREATMENT")
            .otherwise(
                F.when(F.col("TOTAL_NEW_TREATMENTS") < 3, "2_OR_3_NEW_TREATMENTS")
                .otherwise("3_MORE_NEW_TREATMENTS")
            )
        )
    )
)

df_running_active_tx = pivoted_counts(
    df_running_tx_joined,
    NPI_COL,
    "TOTAL_ACTIVE_TREATMENTS",
    "RUNNING_TOTAL_RX_CLAIMS_PATIENT_WITH_",
    ""
)

df_running_new_tx = pivoted_counts(
    df_running_tx_joined,
    NPI_COL,
    "TOTAL_NEW_TREATMENTS",
    "RUNNING_TOTAL_RX_CLAIMS_PATIENT_WITH_",
    ""
)

df_running_patient_treatment = (
    df_running_active_tx
    .join(df_running_new_tx, on=NPI_COL, how="outer")
    .na.fill(0)
)

# ---- Combine all running groups ----
df_running = (
    df_running_total
    .join(df_running_category, on=NPI_COL, how="left")
    .join(df_running_brand, on=NPI_COL, how="left")
    .join(df_running_rejection, on=NPI_COL, how="left")
    .join(df_running_duration, on=NPI_COL, how="left")
    .join(df_running_refill, on=NPI_COL, how="left")
    .join(df_running_payer, on=NPI_COL, how="left")
    .join(df_running_patient, on=NPI_COL, how="left")
    .join(df_running_patient_treatment, on=NPI_COL, how="left")
    .na.fill(0)
)

# ---- Derived running SGLT2_WO_INVOKANA_300MG columns ----
running_sglt2_pairs = [
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_30_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_30_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_30_DAYS_SUPPLY"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_60_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_60_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_60_DAYS_SUPPLY"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_90_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_90_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_90_DAYS_SUPPLY"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_NOT_30_60_90_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_NOT_30_60_90_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_NOT_30_60_90_DAYS_SUPPLY"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_DAYS_SUPPLY_UNKNOWN", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_DAYS_SUPPLY_UNKNOWN", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_DAYS_SUPPLY_UNKNOWN"),
    ("RUNNING_TOTAL_REJECTED_RX_CLAIMS_SGLT2", "RUNNING_TOTAL_REJECTED_RX_CLAIMS_INVOKANA_300MG", "RUNNING_TOTAL_REJECTED_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG"),
]

for age_bin in [
    "AGE_GROUP_0_10", "AGE_GROUP_10_20", "AGE_GROUP_20_30", "AGE_GROUP_30_40",
    "AGE_GROUP_40_50", "AGE_GROUP_50_60", "AGE_GROUP_60_70", "AGE_GROUP_70_80",
    "AGE_GROUP_ABOVE_80"
]:
    running_sglt2_pairs.append((
        f"RUNNING_TOTAL_RX_CLAIMS_SGLT2_{age_bin}",
        f"RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_{age_bin}",
        f"RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_{age_bin}"
    ))

for gender in ["M", "F", "U"]:
    running_sglt2_pairs.append((
        f"RUNNING_TOTAL_RX_CLAIMS_SGLT2_{gender}",
        f"RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_{gender}",
        f"RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_{gender}"
    ))

running_cols = set(df_running.columns)
for sglt2_col, inv_col, out_col in running_sglt2_pairs:
    if sglt2_col in running_cols and inv_col in running_cols:
        df_running = df_running.withColumn(out_col, F.col(sglt2_col) - F.col(inv_col))
    elif sglt2_col in running_cols:
        df_running = df_running.withColumn(out_col, F.col(sglt2_col))

print(f"Running universe: {df_running.count():,} NPIs")
print(f"RUNNING_ cols: {len([c for c in df_running.columns if c.startswith('RUNNING_')])}")

In [0]:
# ---- AVG Features (two-stage monthly aggregation + zero-fill) ----
# This overwrites the earlier direct-window df_avg with logic that mirrors the old notebook:
#   1) aggregate to NPI × calendar month
#   2) zero-fill missing months
#   3) sum monthly values and divide by the fixed denominator
# The denominator is N or N+1 depending on AVG_EXTRA_MONTH.


def build_avg_features_two_stage(df_rx_base, df_rx_pat_base, npi_col, svc_dt_col, avg_window_bounds):
    avg_npi_universe = (
        df_rx_base
        .filter(F.col(npi_col).isNotNull())
        .select(npi_col)
        .distinct()
    )

    window_dfs = []

    for w in sorted(avg_window_bounds.keys()):
        bounds = avg_window_bounds[w]
        suffix = f"_IN_LAST_{w}_MONTH"
        denominator = (w + 1) if AVG_EXTRA_MONTH else w

        # Filter to the window first so partial boundary months respect AVG_WINDOW_BOUNDS.
        df_w = (
            filter_to_window(df_rx_base, svc_dt_col, bounds["start"], bounds["end"])
            .withColumn("_SVC_MONTH", assign_cohort_month(F.col(svc_dt_col)))
        )
        df_w_pat = (
            filter_to_window(df_rx_pat_base, svc_dt_col, bounds["start"], bounds["end"])
            .withColumn("_SVC_MONTH", assign_cohort_month(F.col(svc_dt_col)))
        )

        # Monthly metrics (same per-month semantics as the old monthly notebook).
        df_monthly_age = (
            df_w_pat.groupBy(npi_col, "_SVC_MONTH")
            .agg(F.avg("AGE").alias("AVG_PATIENT_AGE_MONTHLY"))
        )

        df_monthly_pat_count = (
            df_w.groupBy(npi_col, "_SVC_MONTH")
            .agg(F.countDistinct("PATIENT_ID").alias("TOTAL_PATIENT_COUNT_MONTHLY"))
        )

        df_monthly_tx_per_pat = (
            df_w.filter("lower(TREATMENT_CATEGORY) NOT LIKE '%other%'")
            .groupBy(npi_col, "_SVC_MONTH", "PATIENT_ID")
            .agg(F.countDistinct("TREATMENT_CATEGORY").alias("tx_count"))
            .groupBy(npi_col, "_SVC_MONTH")
            .agg(
                F.sum("tx_count").alias("_total_tx"),
                F.countDistinct("PATIENT_ID").alias("_total_pat")
            )
            .withColumn(
                "AVG_TREATMENT_PER_PATIENT_MONTHLY",
                F.when(F.col("_total_pat") != 0, F.col("_total_tx") / F.col("_total_pat"))
                .otherwise(F.lit(0.0))
            )
            .drop("_total_tx", "_total_pat")
        )

        df_monthly = (
            df_monthly_age
            .join(df_monthly_pat_count, on=[npi_col, "_SVC_MONTH"], how="outer")
            .join(df_monthly_tx_per_pat, on=[npi_col, "_SVC_MONTH"], how="outer")
        )

        # Build the month spine so empty months count as zero.
        first_month = (datetime.strptime(bounds["start"], "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-01")
        last_month = datetime.strptime(bounds["end"], "%Y-%m-%d").strftime("%Y-%m-01")

        month_spine = (
            spark.range(1)
            .select(
                F.explode(
                    F.sequence(
                        F.to_date(F.lit(first_month)),
                        F.to_date(F.lit(last_month)),
                        F.expr("interval 1 month")
                    )
                ).alias("_SVC_MONTH")
            )
        )

        df_monthly_filled = (
            avg_npi_universe
            .crossJoin(month_spine)
            .join(df_monthly, on=[npi_col, "_SVC_MONTH"], how="left")
            .na.fill({
                "AVG_PATIENT_AGE_MONTHLY": 0.0,
                "TOTAL_PATIENT_COUNT_MONTHLY": 0.0,
                "AVG_TREATMENT_PER_PATIENT_MONTHLY": 0.0,
            })
        )

        df_window = (
            df_monthly_filled
            .groupBy(npi_col)
            .agg(
                (F.sum("AVG_PATIENT_AGE_MONTHLY") / F.lit(float(denominator))).alias(f"AVG_PATIENT_AGE{suffix}"),
                (F.sum("TOTAL_PATIENT_COUNT_MONTHLY") / F.lit(float(denominator))).alias(f"TOTAL_PATIENT_COUNT{suffix}"),
                (F.sum("AVG_TREATMENT_PER_PATIENT_MONTHLY") / F.lit(float(denominator))).alias(f"AVG_TREATMENT_PER_PATIENT{suffix}")
            )
        )

        window_dfs.append(df_window)

    result = window_dfs[0]
    for df_next in window_dfs[1:]:
        result = result.join(df_next, on=npi_col, how="outer")

    return result.na.fill(0)


# Overwrite df_avg with the monthly zero-fill implementation.
df_avg = build_avg_features_two_stage(df_rx_filtered, df_rx_pat, NPI_COL, SVC_COL, AVG_WINDOW_BOUNDS)
print(f"Avg feature cols (two-stage): {len(df_avg.columns) - 1}")

### 2. Combine All Feature Groups + Derived Features

In [0]:
# ---- Combine all feature groups on NPI ----
# Widen NPI universe: include doctors from full history (df_running)
# who have nonzero RUNNING_ values but no claims in the 13-month window.
# Without this, ~1,200 doctors per cohort would get RUNNING_=0 instead of
# their correct historical counts.
npi_universe_wide = npi_universe.union(df_running.select(NPI_COL)).distinct()
print(f"NPI universe widened: {npi_universe.count():,} → {npi_universe_wide.count():,} "
      f"(+{npi_universe_wide.count() - npi_universe.count():,} from older history)")

df_spine = (
    npi_universe_wide
    .join(df_total, on=NPI_COL, how="left")
    .join(df_category, on=NPI_COL, how="left")
    .join(df_brand, on=NPI_COL, how="left")
    .join(df_rejection, on=NPI_COL, how="left")
    .join(df_duration, on=NPI_COL, how="left")
    .join(df_refill, on=NPI_COL, how="left")
    .join(df_payer, on=NPI_COL, how="left")
    .join(df_patient, on=NPI_COL, how="left")
    .join(df_patient_treatment, on=NPI_COL, how="left")
    .join(df_avg, on=NPI_COL, how="left")
    .join(df_running, on=NPI_COL, how="left")
    .na.fill(0)
)

print(f"Combined spine: {df_spine.count():,} NPIs, {len(df_spine.columns)} columns")

In [0]:
# ---- SGLT2 Without Invokana 300MG (same derived columns as original) ----
# These subtract Invokana 300MG from SGLT2 totals.
# We generate them for each window that has both columns.

sglt2_wo_pairs = [
    # (SGLT2 col pattern, INVOKANA col pattern, output pattern)
    ("TOTAL_RX_CLAIMS_SGLT2", "TOTAL_RX_CLAIMS_INVOKANA_300MG", "TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG"),
    ("TOTAL_RX_CLAIMS_SGLT2_30_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_INVOKANA_300MG_30_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_30_DAYS_SUPPLY"),
    ("TOTAL_RX_CLAIMS_SGLT2_60_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_INVOKANA_300MG_60_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_60_DAYS_SUPPLY"),
    ("TOTAL_RX_CLAIMS_SGLT2_90_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_INVOKANA_300MG_90_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_90_DAYS_SUPPLY"),
    ("TOTAL_REJECTED_RX_CLAIMS_SGLT2", "TOTAL_REJECTED_RX_CLAIMS_INVOKANA_300MG", "TOTAL_REJECTED_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG"),
]

# Also age group and gender variants
for age_bin in ["AGE_GROUP_0_10", "AGE_GROUP_10_20", "AGE_GROUP_20_30", "AGE_GROUP_30_40",
                "AGE_GROUP_40_50", "AGE_GROUP_50_60", "AGE_GROUP_60_70", "AGE_GROUP_70_80",
                "AGE_GROUP_ABOVE_80"]:
    sglt2_wo_pairs.append((
        f"TOTAL_RX_CLAIMS_SGLT2_{age_bin}",
        f"TOTAL_RX_CLAIMS_INVOKANA_300MG_{age_bin}",
        f"TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_{age_bin}"
    ))

for gender in ["M", "F", "U"]:
    sglt2_wo_pairs.append((
        f"TOTAL_RX_CLAIMS_SGLT2_{gender}",
        f"TOTAL_RX_CLAIMS_INVOKANA_300MG_{gender}",
        f"TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_{gender}"
    ))

spine_cols = set(df_spine.columns)
for sglt2_pat, inv_pat, out_pat in sglt2_wo_pairs:
    for w in LOOKBACK_WINDOWS:
        sfx = f"_IN_LAST_{w}_MONTH"
        sglt2_col = f"{sglt2_pat}{sfx}"
        inv_col = f"{inv_pat}{sfx}"
        out_col = f"{out_pat}{sfx}"
        if sglt2_col in spine_cols and inv_col in spine_cols:
            df_spine = df_spine.withColumn(out_col, F.col(sglt2_col) - F.col(inv_col))
        elif sglt2_col in spine_cols:
            # Invokana col missing (no data) — SGLT2 WO = SGLT2
            df_spine = df_spine.withColumn(out_col, F.col(sglt2_col))

print(f"After SGLT2_WO: {len(df_spine.columns)} columns")

In [0]:
# ---- PCT Features: each feature / total claims in the same window ----
pct_exprs = []

for w in LOOKBACK_WINDOWS:
    sfx = f"_IN_LAST_{w}_MONTH"
    denom_col = f"TOTAL_RX_CLAIMS{sfx}"
    
    if denom_col not in df_spine.columns:
        continue
    
    for c in df_spine.columns:
        if c.endswith(sfx) and c != denom_col and c != NPI_COL:
            pct_name = f"PCT_{c}"
            pct_exprs.append(
                F.when(F.col(denom_col) != 0, F.col(c) / F.col(denom_col))
                .otherwise(F.lit(0.0))
                .alias(pct_name)
            )

df_spine = df_spine.select("*", *pct_exprs)
print(f"After PCT: {len(df_spine.columns)} columns")

In [0]:
# ---- Difference features: _IN_LAST_3_6_MONTH = last_6 - last_3 ----
# ---- Ratio features: prior_3 / last_3 ----
# These measure the "prior 3 months" (months 4-6 back) vs the most recent 3.

from pyspark.sql.functions import col, when, lit

def safe_div(numerator_col, denominator_col):
    return when(col(denominator_col) != 0, col(numerator_col) / col(denominator_col)).otherwise(0.0)

# Treatment categories and brands for diff/ratio
diff_ratio_features = [
    "TOTAL_RX_CLAIMS_ACE_OR_ARB",
    "TOTAL_RX_CLAIMS_CALCIUM_CHANNEL_OR_BETA_BLOCKER",
    "TOTAL_RX_CLAIMS_COMBO_DRUG",
    "TOTAL_RX_CLAIMS_DIURETIC_NON_MRA",
    "TOTAL_RX_CLAIMS_INSULIN",
    "TOTAL_RX_CLAIMS_MRA",
    "TOTAL_RX_CLAIMS_SGLT2",
    "TOTAL_RX_CLAIMS_METFORMIN",
    "TOTAL_RX_CLAIMS_FARXIGA",
    "TOTAL_RX_CLAIMS_JARDIANCE",
    "TOTAL_RX_CLAIMS_SPIRONOLACTONE_MRA",
    "TOTAL_RX_CLAIMS",
]

# Only create diff/ratio if both 3m and 6m windows exist
if 3 in LOOKBACK_WINDOWS and 6 in LOOKBACK_WINDOWS:
    for feat in diff_ratio_features:
        col_3m = f"{feat}_IN_LAST_3_MONTH"
        col_6m = f"{feat}_IN_LAST_6_MONTH"
        diff_col = f"{feat}_IN_LAST_3_6_MONTH"
        ratio_col = feat.replace("TOTAL_RX_CLAIMS", "RATIO_RX_CLAIMS") + "_IN_PRIOR_3_LAST_3_MONTH"
        
        if col_3m in df_spine.columns and col_6m in df_spine.columns:
            # Difference: prior 3 months = last 6 - last 3
            df_spine = df_spine.withColumn(diff_col, col(col_6m) - col(col_3m))
            # Ratio: prior 3 / last 3
            df_spine = df_spine.withColumn(ratio_col, safe_div(diff_col, col_3m))

print(f"After Diff/Ratio: {len(df_spine.columns)} columns")

In [0]:
# ---- Add cohort metadata ----
claim_rx_monthly = (
    df_spine
    .withColumn("COHORT_ID", F.lit(COHORT_ID))
    .withColumn("COHORT_MONTH", F.lit(COHORT_MONTH))
    .withColumn("OFFSET_DAYS", F.lit(OFFSET_DAYS))
    .withColumn("CUTOFF_DATE", F.lit(CUTOFF_DATE.strftime("%Y-%m-%d")))
    .withColumn("SVC_DT_M", F.to_timestamp(F.lit(f"{COHORT_MONTH}-01")))
    .withColumn("PULL_DT", F.current_date())
)

# Column count is metadata-only (no Spark action needed)
print(f"Final output columns: {len(claim_rx_monthly.columns)}")
# NOTE: count() and display() removed — each forces a full pipeline re-evaluation
# without .cache(). Row count is verified after the write (cell 26).

In [0]:
# ---- Load written output for tracing ----
# All config variables (COHORT_MONTH, OFFSET_DAYS, CUTOFF_DATE, WINDOW_BOUNDS, etc.)
# are already in scope from 00_cohort_config.
import pandas as pd

_BASE_OUTPUT = (
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
    "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/"
    "claim_rx_monthly_cohort"
)
claim_rx_monthly = spark.read.format("delta").load(_BASE_OUTPUT).filter(F.col("COHORT_ID") == COHORT_ID)

# Load raw claims for tracing
df_rx = spark.read.format("delta").load(
    '/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/'
    'kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/Rx_01_new'
).withColumn('TREATMENT_CATEGORY', F.regexp_replace('TREATMENT_CATEGORY', r'[\s]*$', ''))

# Pick the HCP with the most claims in 12m
top_npi = (
    claim_rx_monthly
    .select("NPI_NUMBER", "TOTAL_RX_CLAIMS_IN_LAST_12_MONTH")
    .orderBy(F.col("TOTAL_RX_CLAIMS_IN_LAST_12_MONTH").desc())
    .first()[0]
)
print(f"Selected NPI: {top_npi}")
hcp_row = claim_rx_monthly.filter(F.col("NPI_NUMBER") == top_npi)

# ---- Raw claims summary ----
print("\n" + "="*70)
print(f"RAW CLAIMS BREAKDOWN for NPI {top_npi}")
print("="*70)

for w in [1, 3, 6, 9, 12]:
    bounds = WINDOW_BOUNDS[w]
    cnt = (
        df_rx.filter(F.col("NPI_NUMBER") == top_npi)
        .filter((F.col("SVC_DT") > F.lit(bounds["start"])) & (F.col("SVC_DT") <= F.lit(bounds["end"])))
        .count()
    )
    print(f"  Window {w:>2}m ({bounds['start']} to {bounds['end']}): {cnt:,} raw claims")

print("\nTreatment Category mix (last 12m):")
df_rx.filter(F.col("NPI_NUMBER") == top_npi).groupBy("TREATMENT_CATEGORY").agg(F.count("*").alias("claims")).orderBy(F.desc("claims")).show(truncate=False)

# ---- Transpose final output ----
pdf = hcp_row.toPandas().T
pdf.columns = ["value"]
pdf = pdf[pdf["value"] != 0]

# Group columns by feature type
def classify_col(col_name):
    if col_name.startswith("COHORT") or col_name.startswith("OFFSET") or col_name in ("CUTOFF_DATE", "SVC_DT_M", "PULL_DT", "NPI_NUMBER"):
        return "0_METADATA"
    elif col_name.startswith("PCT_"):
        return "7_PCT"
    elif col_name.startswith("RATIO_"):
        return "8_RATIO"
    elif "_3_6_MONTH" in col_name:
        return "8_DIFF"
    elif col_name.startswith("RUNNING_"):
        return "6_RUNNING"
    elif col_name.startswith("AVG_") or col_name.startswith("TOTAL_PATIENT_COUNT"):
        return "5_AVG"
    elif "REJECTED" in col_name:
        return "2_REJECTION"
    elif "DAYS_SUPPLY" in col_name:
        return "3_DURATION"
    elif "REFILL" in col_name or "NEW_PRESCRIPTION" in col_name:
        return "3_REFILL"
    elif "METHOD_OF_PAYMENT" in col_name:
        return "4_PAYER"
    elif "AGE_GROUP" in col_name or "PATIENT_GENDER" in col_name:
        return "4_PATIENT_ATTR"
    elif "ACTIVE_TREATMENT" in col_name or "NEW_TREATMENT" in col_name:
        return "5_PATIENT_TX"
    elif "SGLT2_WO" in col_name:
        return "2_SGLT2_WO"
    elif col_name.startswith("TOTAL_RX_CLAIMS_IN_LAST"):
        return "1_TOTAL"
    else:
        return "1_TREATMENT_BRAND"

pdf["group"] = [classify_col(idx) for idx in pdf.index]

print("\n" + "="*70)
print(f"FINAL OUTPUT — NON-ZERO FEATURES for NPI {top_npi}")
print(f"Cohort: {COHORT_MONTH} | Offset: {OFFSET_DAYS}d | Cutoff: {CUTOFF_DATE.date()}")
print(f"Total columns: {len(claim_rx_monthly.columns):,} | Non-zero: {len(pdf):,}")
print("="*70)

for group_name in sorted(pdf["group"].unique()):
    group_df = pdf[pdf["group"] == group_name].drop(columns=["group"])
    label = group_name.split("_", 1)[1]
    print(f"\n--- {label} ({len(group_df)} features) ---")
    for idx, row in group_df.iterrows():
        val = row["value"]
        if isinstance(val, float):
            print(f"  {idx:<80s} {val:>12.4f}")
        else:
            print(f"  {idx:<80s} {val:>12}")

### 3. Write Output

In [0]:
# ---- Write output ----
# Partitioned by COHORT_ID so multiple cohorts can coexist.
OUTPUT_PATH = (
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
    "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/"
    "claim_rx_monthly_cohort"
)

claim_rx_monthly \
    .write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("replaceWhere", f"COHORT_ID = '{COHORT_ID}'") \
    .partitionBy("COHORT_ID") \
    .save(OUTPUT_PATH)

print(f"Written to {OUTPUT_PATH} (partition: COHORT_ID = {COHORT_ID})")

### 4. Validation — Offset = 0 Compatibility Check

At offset = 0 the window bounds snap to calendar-month boundaries, so the new output should match the old `claim_rx_monthly` exactly (no day-level rounding differences). Any mismatch indicates a bug.

In [0]:
# ---- Validation: Compare with original output at offset = 0 ----
# Run after executing the full notebook with OFFSET_DAYS = 0.
# At offset 0 the window bounds snap to calendar months, so values should match exactly.

original = spark.read.format("delta").load(
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
    "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/claim_rx_monthly"
)

# Pick 10 sample NPIs with activity in the cohort month
sample_npis = [
    r.NPI_NUMBER for r in
    original
    .filter(F.col("SVC_DT_M") == F.lit(f"{COHORT_MONTH}-01"))
    .select("NPI_NUMBER")
    .limit(10)
    .collect()
]
print(f"Validating {len(sample_npis)} NPIs for {COHORT_MONTH}\n")

# Find common non-key columns
old_cols = set(original.columns)
new_cols = set(claim_rx_monthly.columns)
common_feature_cols = sorted(
    (old_cols & new_cols) - {"NPI_NUMBER", "SVC_DT_M", "COHORT_ID", "COHORT_MONTH",
                             "OFFSET_DAYS", "CUTOFF_DATE", "PULL_DT", "PROVIDER_ID"}
)

mismatch_count = 0
checked = 0
for npi in sample_npis:
    old_row = (
        original.filter(
            (F.col("NPI_NUMBER") == npi) &
            (F.col("SVC_DT_M") == F.lit(f"{COHORT_MONTH}-01"))
        )
    )
    new_row = claim_rx_monthly.filter(F.col("NPI_NUMBER") == npi)

    if old_row.count() == 0 or new_row.count() == 0:
        continue

    old_vals = old_row.select(common_feature_cols).first().asDict()
    new_vals = new_row.select(common_feature_cols).first().asDict()

    for c in common_feature_cols:
        checked += 1
        ov, nv = old_vals.get(c), new_vals.get(c)
        if ov != nv:
            mismatch_count += 1
            if mismatch_count <= 20:
                print(f"\u26a0\ufe0f  NPI={npi}  {c}: old={ov}  new={nv}")

print(f"\nChecked {checked:,} cells across {len(sample_npis)} NPIs, {len(common_feature_cols)} common columns.")
print(f"Mismatches: {mismatch_count}")
if mismatch_count == 0:
    print("\u2705 All values match.")

In [0]:
# ---- Column Diff: old claim_rx_monthly vs new claim_rx_monthly_cohort ----

old_output = spark.read.format("delta").load(
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
    "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/claim_rx_monthly"
)

old_cols = set(old_output.columns)
new_cols = set(claim_rx_monthly.columns)

old_only  = sorted(old_cols - new_cols)
new_only  = sorted(new_cols - old_cols)
common    = sorted(old_cols & new_cols)

# ---- Classify by feature type ----
def classify_col(c):
    if c in ("NPI_NUMBER", "SVC_DT_M", "COHORT_ID", "COHORT_MONTH",
            "OFFSET_DAYS", "CUTOFF_DATE", "PULL_DT", "PROVIDER_ID"):
        return "metadata"
    if c.startswith("RUNNING_"):
        return "RUNNING"
    if c.startswith("PCT_"):
        return "PCT"
    if c.startswith("AVG_") or c.startswith("TOTAL_PATIENT_COUNT"):
        return "AVG"
    if c.startswith("RATIO_"):
        return "RATIO"
    if "_IN_LAST_3_6_MONTH" in c:
        return "DIFF"
    if "_IN_LAST_" in c:
        return "windowed_SUM"
    return "other"

from collections import Counter

print(f"Old output: {len(old_cols):,} cols")
print(f"New output: {len(new_cols):,} cols")
print(f"Common:     {len(common):,}")
print(f"Old-only:   {len(old_only):,}")
print(f"New-only:   {len(new_only):,}")

print("\n--- Old-only columns by feature type ---")
for ftype, cnt in sorted(Counter(classify_col(c) for c in old_only).items(), key=lambda x: -x[1]):
    print(f"  {ftype:15s} {cnt:5d}")
    if cnt <= 10:
        for c in old_only:
            if classify_col(c) == ftype:
                print(f"    {c}")

print("\n--- New-only columns by feature type ---")
for ftype, cnt in sorted(Counter(classify_col(c) for c in new_only).items(), key=lambda x: -x[1]):
    print(f"  {ftype:15s} {cnt:5d}")
    if cnt <= 10:
        for c in new_only:
            if classify_col(c) == ftype:
                print(f"    {c}")

# ---- Model features check ----
# The model uses 172 features from this notebook:
#   52 RUNNING_, 63 windowed SUM, 56 PCT, 1 RATIO (JARDIANCE), 0 AVG.
MODEL_FEATURE_COUNTS = {"RUNNING": 52, "windowed_SUM": 63, "PCT": 56, "RATIO": 1}
print("\n--- Model feature coverage ---")
total_model = 0
total_present = 0
for ftype, expected in MODEL_FEATURE_COUNTS.items():
    present = [c for c in new_cols if classify_col(c) == ftype]
    total_model += expected
    total_present += builtins.min(len(present), expected)
    status = "\u2705" if len(present) >= expected else "\u26a0\ufe0f"
    print(f"  {status} {ftype:15s}: {len(present):4d} available, {expected} needed by model")

print(f"\nTotal model features: {total_model}")
print(f"Covered: {total_present} / {total_model}")
if total_present >= total_model:
    print("\u2705 All 172 model features have sufficient coverage.")
else:
    print("\u26a0\ufe0f Some model features may be missing — check the lists above.")

In [0]:
# ---- Step 5: Test offsets 7, 14, 21 on sample NPIs by hand ----
# Pick a few active NPIs and compare claim-by-claim which ones fall inside
# each window at different offsets.

from dateutil.relativedelta import relativedelta
from datetime import datetime, timedelta
import pyspark.sql.functions as F

TEST_COHORT = "2025-06"  # Use June (a 30-day month, boundary-bug affected)
TEST_OFFSETS = [0, 7, 14, 21]
TEST_WINDOW = 3  # months
N_SAMPLE_NPIS = 3

df_rx_raw = spark.read.format("delta").load(
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
    "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/Rx_01_new"
)

# Pick NPIs with claims in the test period
_cs = datetime.strptime(f"{TEST_COHORT}-01", "%Y-%m-%d")
_me = _cs + relativedelta(months=1) - timedelta(days=1)

active_npis = (
    df_rx_raw
    .filter((F.col("SVC_DT") >= F.lit((_cs - relativedelta(months=12)).strftime("%Y-%m-%d"))) &
            (F.col("SVC_DT") <= F.lit((_me + timedelta(days=21)).strftime("%Y-%m-%d"))))
    .groupBy("NPI_NUMBER")
    .agg(F.count("*").alias("n_claims"),
         F.countDistinct(F.date_trunc("MM", F.col("SVC_DT"))).alias("n_months"))
    .filter("n_claims BETWEEN 10 AND 50 AND n_months >= 4")
    .limit(N_SAMPLE_NPIS)
    .select("NPI_NUMBER")
    .collect()
)
sample_npis = [r.NPI_NUMBER for r in active_npis]
print(f"Sample NPIs: {sample_npis}\n")

for npi in sample_npis:
    print(f"\n{'='*80}")
    print(f"NPI: {npi}")
    
    # Get all claims for this NPI in the broad range
    claims = (
        df_rx_raw
        .filter(F.col("NPI_NUMBER") == npi)
        .filter((F.col("SVC_DT") >= F.lit((_cs - relativedelta(months=12)).strftime("%Y-%m-%d"))) &
                (F.col("SVC_DT") <= F.lit((_me + timedelta(days=21)).strftime("%Y-%m-%d"))))
        .select("SVC_DT", "TREATMENT_CATEGORY", "PATIENT_ID")
        .orderBy("SVC_DT")
        .collect()
    )
    
    print(f"  Claims in broad window: {len(claims)}")
    print(f"  {'SVC_DT':<12} {'TREATMENT_CATEGORY':<30}", end="")
    for offset in TEST_OFFSETS:
        print(f" off={offset:<3}", end="")
    print()
    print(f"  {'-'*12} {'-'*30}", end="")
    for _ in TEST_OFFSETS:
        print(f" {'-'*7}", end="")
    print()
    
    offset_counts = {o: 0 for o in TEST_OFFSETS}
    
    for row in claims:
        svc = row.SVC_DT
        tx = row.TREATMENT_CATEGORY or "(null)"
        print(f"  {str(svc):<12} {tx:<30}", end="")
        
        for offset in TEST_OFFSETS:
            cutoff = _me + timedelta(days=offset)
            if offset == 0:
                w_start = _cs - relativedelta(months=TEST_WINDOW - 1) - timedelta(days=1)
            else:
                w_start = cutoff - relativedelta(months=TEST_WINDOW)
            
            in_window = svc > w_start.date() and svc <= cutoff.date()
            mark = "  ✓  " if in_window else "  ·  "
            if in_window:
                offset_counts[offset] += 1
            print(f"  {mark}", end="")
        print()
    
    print(f"  {'TOTAL':<12} {'last ' + str(TEST_WINDOW) + 'm claim count':<30}", end="")
    for offset in TEST_OFFSETS:
        print(f"  {offset_counts[offset]:>3}  ", end="")
    print()

print(f"\n\nExpected: offset 0 matches old notebook; higher offsets shift the window forward.")
print(f"Check: do any claims land inside one offset's window but not another's as expected?")

## Part 4 -- Status and Flags for Wenjie

### Resolved in This Notebook

* **`RUNNING_` features** are now included. A dedicated cell counts all claims from the start of `Rx_01_new` history through `CUTOFF_DATE` per NPI, with null NPIs excluded. It covers every feature group the old notebook had (treatment category, brand, rejection, duration, brand x refill, refill alone, payer, patient attributes, patient treatment buckets) plus the `SGLT2_WO_INVOKANA_300MG` derived running columns.
* **AVG extra-month semantics** are controlled by `AVG_EXTRA_MONTH` (default `True` = old behavior). A two-stage cell aggregates per NPI per calendar month, zero-fills missing months, then divides by the fixed denominator (N+1 when True, N when False). This replicates the old `step3_lag_avg` behavior exactly.
* **Treatment x refill columns** removed from the refill group. Old notebook had brand x refill + refill alone only; the extra treatment x refill columns were new and are now gone.
* **Window boundary snap at offset 0**: `_build_bounds` snaps exclusive start to `first_of(cohort_month - (N-1 months)) - 1 day`, so windows align exactly with calendar months. No 1-2 day rounding differences.
* **`.cache()` removed**: Not supported on serverless.
* **`import builtins`** guards against `00_connection` shadowing Python `max`/`min`/`sum`.

### Still Open -- Discuss with Wenjie

1. ~~**`pat_new_and_total_treatments` month alignment**~~ **RESOLVED**: Added `assign_cohort_month()` helper (mirrors `00_cohort_config`) and replaced all `date_trunc('MM', ...)` with it in cells 18, 19, and 20. At offset>0, claims now map to the same cohort month as the patient journey pipeline, so the join is correct.

2. **NPI to BH_ID mapping**: The original notebook joins this at the end (cells 73-74). Not yet added to this notebook. Add when the mapping table path is confirmed.

3. **`99_Combine_Physician_Data` wiring**: The combine notebook joins on `SVC_DT_M`. This notebook outputs `COHORT_ID` instead. Decide how to bridge: (a) derive `SVC_DT_M` from cohort config, or (b) switch the join key.

4. **`00_step23_function`** helpers (`step2_fill_missing`, `step3_lag`, `step3_lag_avg`) are **not used** here. Other notebooks (02-06) still depend on them.

### Nice to Have

5. **Multi-cohort generation**: One cohort per run. For training, extend with a parameterized Lakeflow Job.
6. **Patient journey provenance**: Locate where the claim-eligible patient file is created upstream.
7. **Columns from old LAD V6**: Cross-check column lists when validating in case of missing switch/add-on columns.

In [0]:
# ============================================================
# HCP FEATURE TRACE — End-to-end validation for one test NPI
# ============================================================
# Pick the test NPI with the most claims so we have rich data to verify.

TRACE_NPIS = get_test_npis()
df_test = df_rx_filtered.filter(F.col("NPI_NUMBER").isin(TRACE_NPIS))

# Find busiest NPI
npi_summary = (
    df_test.groupBy("NPI_NUMBER").agg(
        F.count("*").alias("total_claims"),
        F.countDistinct("PATIENT_ID").alias("unique_patients"),
        F.countDistinct("TREATMENT_CATEGORY").alias("tx_categories"),
        F.sum(F.when(F.col("REJECTION_FLAG") == 1, 1).otherwise(0)).alias("rejected_claims"),
        F.min("SVC_DT").alias("first_claim"),
        F.max("SVC_DT").alias("last_claim")
    ).orderBy(F.desc("total_claims"))
)
display(npi_summary)

# Pick the top NPI
TRACE_NPI = npi_summary.first()["NPI_NUMBER"]
print(f"\n>>> Tracing NPI: {TRACE_NPI}")

# ---- Raw claims per window ----
print(f"\n{'='*70}")
print(f"RAW CLAIMS PER LOOKBACK WINDOW (Cutoff: {CUTOFF_DATE.strftime('%Y-%m-%d')})")
print(f"{'='*70}")

df_npi = df_rx_filtered.filter(F.col("NPI_NUMBER") == TRACE_NPI)

for w in LOOKBACK_WINDOWS:
    wb = WINDOW_BOUNDS[w]
    ab = AVG_WINDOW_BOUNDS[w]
    cnt_w = df_npi.filter((F.col("SVC_DT") > F.lit(wb["start"])) & (F.col("SVC_DT") <= F.lit(wb["end"]))).count()
    cnt_a = df_npi.filter((F.col("SVC_DT") > F.lit(ab["start"])) & (F.col("SVC_DT") <= F.lit(ab["end"]))).count()
    print(f"  Window {w:2d}m ({wb['start']} to {wb['end']}): {cnt_w:,} claims")
    print(f"  AVG    {w:2d}m ({ab['start']} to {ab['end']}): {cnt_a:,} claims")

# ---- Treatment category breakdown (last 3m) ----
w3 = WINDOW_BOUNDS[3]
print(f"\n--- Treatment Categories (last 3m: {w3['start']} to {w3['end']}) ---")
df_npi_3m = df_npi.filter((F.col("SVC_DT") > F.lit(w3["start"])) & (F.col("SVC_DT") <= F.lit(w3["end"])))
df_npi_3m.groupBy("TREATMENT_CATEGORY").agg(
    F.count("*").alias("claims"),
    F.countDistinct("PATIENT_ID").alias("patients")
).orderBy(F.desc("claims")).show(50, truncate=False)

# ---- Brand breakdown (last 3m) ----
print(f"--- Brand Categories (last 3m, BRAND_FILTER only) ---")
df_npi_3m.filter(BRAND_FILTER).groupBy("Category").agg(
    F.count("*").alias("claims")
).orderBy(F.desc("claims")).show(20, truncate=False)

In [0]:
# ============================================================
# TRACE STEP 2: Verify each feature group against manual counts
# ============================================================
# We'll recompute key features for TRACE_NPI by hand and compare
# with what the feature-building functions produce.

import pandas as pd

def trace_feature_group(label, df_feature, expected_counts, npi=TRACE_NPI):
    """Compare feature group output vs manually counted expected values."""
    row = df_feature.filter(F.col(NPI_COL) == npi)
    if row.count() == 0:
        print(f"  [{label}] NPI {npi} NOT FOUND in feature group!")
        return
    pdf = row.toPandas().T
    pdf.columns = ["value"]
    pdf = pdf[pdf.index != NPI_COL]
    
    print(f"\n{'='*70}")
    print(f"FEATURE GROUP: {label} — NPI {npi}")
    print(f"{'='*70}")
    
    mismatches = 0
    for col_name, expected_val in expected_counts.items():
        if col_name in pdf.index:
            actual = pdf.loc[col_name, "value"]
            match = "✅" if actual == expected_val else "❌"
            if actual != expected_val:
                mismatches += 1
            print(f"  {match} {col_name}: expected={expected_val}, actual={actual}")
        else:
            print(f"  ❌ {col_name}: MISSING from output")
            mismatches += 1
    
    print(f"\n  Total features: {len(pdf)} | Checked: {len(expected_counts)} | Mismatches: {mismatches}")
    return pdf

# ---- 2.1 Total Rx Claims (denominator) ----
# Manually count
manual_total = {}
for w in LOOKBACK_WINDOWS:
    wb = WINDOW_BOUNDS[w]
    cnt = df_npi.filter((F.col("SVC_DT") > F.lit(wb["start"])) & (F.col("SVC_DT") <= F.lit(wb["end"]))).count()
    manual_total[f"TOTAL_RX_CLAIMS_IN_LAST_{w}_MONTH"] = cnt

df_total = build_total_col(df_rx_filtered, NPI_COL, SVC_COL, WINDOW_BOUNDS)
trace_feature_group("TOTAL (denominator)", df_total, manual_total)

# ---- 2.1 Treatment Category ----
# Manually count SGLT2 in last 3m
w3 = WINDOW_BOUNDS[3]
sglt2_3m = df_npi.filter(
    (F.col("SVC_DT") > F.lit(w3["start"])) & (F.col("SVC_DT") <= F.lit(w3["end"])) &
    (F.col("TREATMENT_CATEGORY") == "SGLT2")
).count()

df_category = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col="TREATMENT_CATEGORY", prefix="TOTAL_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS
)
trace_feature_group("TREATMENT CATEGORY (spot check SGLT2 3m)", df_category,
                    {"TOTAL_RX_CLAIMS_SGLT2_IN_LAST_3_MONTH": sglt2_3m})

# ---- 2.3 Rejection ----
rej_3m = df_npi.filter(
    (F.col("SVC_DT") > F.lit(w3["start"])) & (F.col("SVC_DT") <= F.lit(w3["end"])) &
    (F.col("REJECTION_FLAG") == 1)
).count()

df_rej_category = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col="TREATMENT_CATEGORY", prefix="TOTAL_REJECTED_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS, extra_filter="REJECTION_FLAG = 1"
)
df_rej_brand = build_feature_group(
    df_rx_filtered, NPI_COL, SVC_COL,
    pivot_col="Category", prefix="TOTAL_REJECTED_RX_CLAIMS_",
    window_bounds=WINDOW_BOUNDS,
    extra_filter=f"REJECTION_FLAG = 1 AND {BRAND_FILTER}"
)
df_rejection = df_rej_category.join(df_rej_brand, on=NPI_COL, how="outer").na.fill(0)

# Get all rejected claims by category for 3m
rej_by_cat = (
    df_npi.filter(
        (F.col("SVC_DT") > F.lit(w3["start"])) & (F.col("SVC_DT") <= F.lit(w3["end"])) &
        (F.col("REJECTION_FLAG") == 1)
    ).groupBy("TREATMENT_CATEGORY").count().collect()
)
rej_expected = {f"TOTAL_REJECTED_RX_CLAIMS_{clean_col_name(r['TREATMENT_CATEGORY'])}_IN_LAST_3_MONTH": r['count'] for r in rej_by_cat}
rej_expected["_TOTAL_REJECTED_3M"] = rej_3m  # just for display

print(f"\nRejected claims in last 3m: {rej_3m} total")
for k, v in sorted(rej_expected.items()):
    if not k.startswith("_"):
        print(f"  {k}: {v}")

trace_feature_group("REJECTION (by category, 3m)", df_rejection, 
                    {k: v for k, v in rej_expected.items() if not k.startswith("_")})

In [0]:
# ============================================================
# TRACE STEP 3: Patient Treatment join (assign_cohort_month)
# ============================================================
# This is the critical join — raw claims × pat_new_and_total_treatments
# using assign_cohort_month() to align months at offset=7.

print(f"{'='*70}")
print(f"PATIENT TREATMENT JOIN TRACE — NPI {TRACE_NPI}")
print(f"Offset: {OFFSET_DAYS} | assign_cohort_month maps dates to offset-aligned months")
print(f"{'='*70}")

# Show how claims near month boundaries map
w3 = WINDOW_BOUNDS[3]
df_npi_3m = df_npi.filter(
    (F.col("SVC_DT") > F.lit(w3["start"])) & (F.col("SVC_DT") <= F.lit(w3["end"]))
)

# Add the cohort month column so we can see the mapping
df_npi_3m_mapped = df_npi_3m.withColumn("COHORT_MONTH_MAPPED", assign_cohort_month(F.col("SVC_DT")))

print("\n--- Claims near month boundaries (showing offset alignment) ---")
df_npi_3m_mapped.select(
    "PATIENT_ID", "SVC_DT", "COHORT_MONTH_MAPPED", "TREATMENT_CATEGORY"
).orderBy("SVC_DT").show(30, truncate=False)

# Show the join result
print("--- Joined with pat_new_and_total_treatments ---")
df_joined = df_npi_3m_mapped.join(
    pat_new_and_total_treatments,
    (df_npi_3m_mapped["PATIENT_ID"] == pat_new_and_total_treatments["PATIENT_ID"]) &
    (df_npi_3m_mapped["COHORT_MONTH_MAPPED"] == pat_new_and_total_treatments["SVC_DT_M"]),
    how="left"
).select(
    df_npi_3m_mapped["PATIENT_ID"],
    "SVC_DT", "COHORT_MONTH_MAPPED",
    "TOTAL_TREATMENTS", "TOTAL_NEW_TREATMENTS",
    "TREATMENT_CATEGORY"
)

# Check for nulls in the join (would indicate misaligned months)
null_count = df_joined.filter(F.col("TOTAL_TREATMENTS").isNull()).count()
total_count = df_joined.count()
match_rate = ((total_count - null_count) / total_count * 100) if total_count > 0 else 0

df_joined.orderBy("SVC_DT").show(30, truncate=False)
print(f"Join match rate: {total_count - null_count}/{total_count} ({match_rate:.1f}%)")
if null_count > 0:
    print(f"⚠️  {null_count} claims had no matching patient treatment record")
else:
    print("✅ All claims matched a patient treatment record")

# ============================================================
# TRACE STEP 4: RUNNING totals (full history)
# ============================================================
print(f"\n{'='*70}")
print(f"RUNNING TOTALS — NPI {TRACE_NPI} (all history through {CUTOFF_DATE.strftime('%Y-%m-%d')})")
print(f"{'='*70}")

cutoff_lit = F.to_date(F.lit(CUTOFF_DATE.strftime("%Y-%m-%d")))
df_npi_full = df_rx.filter(
    (F.col("NPI_NUMBER") == TRACE_NPI) &
    (F.col("SVC_DT") <= cutoff_lit)
)

running_total = df_npi_full.count()
print(f"  Total claims (all history): {running_total:,}")

# By treatment category
print("\n  Running totals by Treatment Category:")
df_npi_full.groupBy("TREATMENT_CATEGORY").agg(
    F.count("*").alias("running_count")
).orderBy(F.desc("running_count")).show(20, truncate=False)

# Compare with what build_running_total would produce
df_running_check = (
    df_rx.filter(F.col("NPI_NUMBER").isNotNull())
    .filter(F.col("SVC_DT") <= cutoff_lit)
    .filter(F.col("NPI_NUMBER") == TRACE_NPI)
    .count()
)
status = "✅" if df_running_check == running_total else "❌"
print(f"\n  {status} Running total cross-check: manual={running_total:,}, function input={df_running_check:,}")

In [0]:
# ============================================================
# TRACE STEP 5: Zoom into one patient — before & after the join
# ============================================================
# Pick a patient with multiple claims across months for TRACE_NPI,
# so we can see the month mapping and join clearly.

w3 = WINDOW_BOUNDS[3]
df_npi_3m = df_rx_filtered.filter(
    (F.col("NPI_NUMBER") == TRACE_NPI) &
    (F.col("SVC_DT") > F.lit(w3["start"])) &
    (F.col("SVC_DT") <= F.lit(w3["end"]))
)

# Find a patient with claims in multiple months
patient_pick = (
    df_npi_3m
    .withColumn("_month", assign_cohort_month(F.col("SVC_DT")))
    .groupBy("PATIENT_ID")
    .agg(
        F.count("*").alias("n_claims"),
        F.countDistinct("_month").alias("n_months")
    )
    .filter("n_months >= 2")
    .orderBy(F.desc("n_claims"))
    .first()
)
TRACE_PATIENT = patient_pick["PATIENT_ID"]
print(f"Tracing Patient: {TRACE_PATIENT} ({patient_pick['n_claims']} claims across {patient_pick['n_months']} months)")
print(f"HCP: {TRACE_NPI} | Window: {w3['start']} to {w3['end']} | Offset: {OFFSET_DAYS}\n")

# ============================================================
# TABLE 1: Raw claims (from df_rx) — what the HCP notebook sees
# ============================================================
print("=" * 70)
print("TABLE 1: Raw Rx Claims (df_rx_filtered)")
print("  Grain: one row per claim (HCP × Patient × Date)")
print("=" * 70)
df_patient_claims = (
    df_npi_3m
    .filter(F.col("PATIENT_ID") == TRACE_PATIENT)
    .select("NPI_NUMBER", "PATIENT_ID", "SVC_DT", "TREATMENT_CATEGORY",
            "CATEGORY", "REJECTION_FLAG", "METHOD_OF_PAYMENT", "REFILL_FLAG")
    .orderBy("SVC_DT")
)
print(f"  Rows: {df_patient_claims.count()}")
display(df_patient_claims)

# ============================================================
# TABLE 1b: Same claims with cohort month mapped
# ============================================================
print("\n" + "=" * 70)
print("TABLE 1b: Raw Claims + assign_cohort_month(SVC_DT)")
print(f"  offset={OFFSET_DAYS}: dates before the 7th map to the PREVIOUS month")
print("=" * 70)
df_patient_mapped = (
    df_patient_claims
    .withColumn("COHORT_MONTH_MAPPED", assign_cohort_month(F.col("SVC_DT")))
    .withColumn("PLAIN_TRUNC", F.trunc(F.col("SVC_DT"), "month"))
    .select("PATIENT_ID", "SVC_DT", "PLAIN_TRUNC", "COHORT_MONTH_MAPPED",
            "TREATMENT_CATEGORY")
    .orderBy("SVC_DT")
)
display(df_patient_mapped)

# ============================================================
# TABLE 2: Patient treatment record (from pat_new_and_total_treatments)
# ============================================================
print("\n" + "=" * 70)
print("TABLE 2: pat_new_and_total_treatments (from NB06 patient journey)")
print("  Grain: one row per Patient × Month")
print("  This is what the join brings in — treatment counts per month")
print("=" * 70)
df_patient_treatments = (
    pat_new_and_total_treatments
    .filter(F.col("PATIENT_ID") == TRACE_PATIENT)
    .orderBy("SVC_DT_M")
)
print(f"  Rows: {df_patient_treatments.count()}")
display(df_patient_treatments)

# ============================================================
# TABLE 3: Joined result — claim enriched with treatment counts
# ============================================================
print("\n" + "=" * 70)
print("TABLE 3: After join (claim × patient treatment)")
print("  Join keys: PATIENT_ID + assign_cohort_month(SVC_DT) = SVC_DT_M")
print("  Each claim now carries the patient's treatment profile for that month")
print("=" * 70)
df_patient_joined = (
    df_npi_3m
    .filter(F.col("NPI_NUMBER") == TRACE_NPI)
    .filter(F.col("PATIENT_ID") == TRACE_PATIENT)
    .withColumn("COHORT_MONTH_MAPPED", assign_cohort_month(F.col("SVC_DT")))
    .alias("c")
    .join(
        pat_new_and_total_treatments.alias("p"),
        (F.col("c.PATIENT_ID") == F.col("p.PATIENT_ID")) &
        (F.col("c.COHORT_MONTH_MAPPED") == F.col("p.SVC_DT_M")),
        how="left"
    )
    .select(
        F.col("c.PATIENT_ID"),
        "SVC_DT", "COHORT_MONTH_MAPPED",
        F.col("p.SVC_DT_M").alias("PAT_SVC_DT_M"),
        "TOTAL_TREATMENTS", "TOTAL_NEW_TREATMENTS",
        "TREATMENT_CATEGORY"
    )
    .orderBy("SVC_DT")
)

null_joins = df_patient_joined.filter(F.col("TOTAL_TREATMENTS").isNull()).count()
total_joins = df_patient_joined.count()
display(df_patient_joined)

if null_joins == 0:
    print(f"\n\u2705 All {total_joins} claims matched a patient treatment record")
else:
    print(f"\n\u26a0\ufe0f {null_joins}/{total_joins} claims had no match")
    print("  Unmatched claims:")
    display(df_patient_joined.filter(F.col("TOTAL_TREATMENTS").isNull()))